# 3. Technical Traceability Testing

This notebook validates the **Technical Traceability** prompt used by ClearSpec AI.

The objective is to verify that every generated User Story is mapped to implementation-ready technical artifacts.

For each story, the generated response should include:

- Database schema changes
- REST API endpoints
- Pseudocode for business logic

The production prompt templates are defined in:

- `backend/prompts.py`
- `TRACE_SYSTEM`
- `trace_user_msg(...)`

The generated output should satisfy the structural checklist at the end of this notebook.

In [ ]:
import sys
import re
import asyncio

sys.path.insert(0, "../backend")

from dotenv import load_dotenv

load_dotenv("../backend/.env")

from llm_client import call_llm
from prompts import TRACE_SYSTEM, trace_user_msg

In [ ]:
STORIES = """
## User Stories

### Story 1: View Lab Results

As a doctor,
I want to view consolidated laboratory results for a patient,
so that I can diagnose more quickly.

Acceptance Criteria:

Given a doctor is authenticated,

When they open a patient profile,

Then all laboratory results from the last 90 days load within 2 seconds.

---

### Story 2: Critical Value Paging

As an on-call physician,

I want to receive an alert within 60 seconds whenever a critical laboratory value is reported,

So that urgent treatment can begin immediately.

Acceptance Criteria:

Given a laboratory result is marked critical,

When the result is published,

Then the on-call physician receives both an SMS and a push notification within 60 seconds.

---

### Story 3: Notification Preferences

As a patient,

I want to choose Email or SMS notifications,

So that I receive results using my preferred communication channel.

Acceptance Criteria:

Given a patient opens notification settings,

When they choose a preferred channel,

Then future notifications are delivered using that channel.

Default notification method is Email.
"""

In [ ]:
async def evaluate_trace_prompt():

    print("=" * 80)
    print("Running Technical Traceability Prompt")
    print("=" * 80)

    try:

        artifacts = await call_llm(
            TRACE_SYSTEM,
            trace_user_msg(STORIES)
        )

        print(artifacts)

        print()
        print("=" * 80)
        print("Running Structural Validation")
        print("=" * 80)

        assert (
            "```sql" in artifacts.lower()
            or "create table" in artifacts.lower()
        ), "No SQL schema detected."

        assert (
            "GET " in artifacts
            or "POST " in artifacts
            or "PUT " in artifacts
            or "DELETE " in artifacts
        ), "No REST API endpoints detected."

        assert (
            artifacts.lower().count("story") >= 3
        ), "Not every story appears to have technical mappings."

        print("✓ SQL schema detected.")
        print("✓ REST endpoints detected.")
        print("✓ All stories traced.")
        print()
        print("✓ Structural validation passed.")

    except Exception as e:
        print("Evaluation failed:")
        print(e)

    print()
    print("=" * 80)
    print("Traceability evaluation complete.")

In [ ]:
asyncio.run(evaluate_trace_prompt())

# Output Evaluation Checklist

Verify every generated User Story contains implementation-ready artifacts.

---

## Database Design

- [ ] Database schema changes included
- [ ] SQL statements generated
- [ ] Tables/columns clearly identified

---

## REST API Design

- [ ] GET endpoint generated
- [ ] POST endpoint generated (if required)
- [ ] Proper request/response descriptions included

---

## Business Logic

- [ ] Pseudocode provided
- [ ] Logic is implementation-ready
- [ ] Edge cases considered

---

## Traceability

- [ ] Story 1 traced
- [ ] Story 2 traced
- [ ] Story 3 traced

---

## Overall Score

Prompt Version: __________

Technical Coverage: ____ / 10

Artifact Quality: ____ / 10

Production Ready:

- [ ] Yes
- [ ] Needs Improvement